In [10]:
# 1. Leer DataCruda y quitar únicamente columnas.
from pathlib import Path
import re
import pandas as pd
from IPython.display import display

ruta = Path("DataCruda.xlsx")
HOJA = "Productos"
UMBRAL_VACIOS = 0.80
protegidas = {"SKU", "Nombre_producto", "Descripcion_corta", "Precio_actual", "Marca", "URL_producto", "Palabras_clave"}

if not ruta.is_file():
    raise FileNotFoundError(f"Coloca DataCruda.xlsx en esta carpeta: {Path.cwd()}")
original = pd.read_excel(ruta, sheet_name=HOJA, dtype={"SKU": str}, keep_default_na=False)
if original.empty:
    raise ValueError(f"La hoja {HOJA} no tiene filas de datos.")
if not 0 <= UMBRAL_VACIOS <= 1:
    raise ValueError("El umbral de vacíos debe estar entre 0 y 1.")

revision_columnas = []
eliminar = []
for columna in original.columns:
    texto = original[columna].astype("string").fillna("").str.strip()
    vacios = texto.eq("").mean()
    con_link = texto.str.contains(r"https?://|www\.", case=False, regex=True).any()
    nombre_de_link = bool(re.search(r"(?:^|[ _])(?:urls?|links?|enlaces?)(?:$|[ _])", str(columna), re.I))
    motivos = []
    if nombre_de_link or con_link:
        motivos.append("Contiene enlaces o su encabezado indica enlaces")
    if vacios >= UMBRAL_VACIOS:
        motivos.append(f"Al menos {UMBRAL_VACIOS:.0%} de vacíos")
    if columna in protegidas:
        decision, motivo = "Conservar", "Campo protegido del comparador"
    elif motivos:
        decision, motivo = "Eliminar", " / ".join(motivos)
        eliminar.append(columna)
    else:
        decision, motivo = "Conservar", "Columna con datos"
    revision_columnas.append([columna, round(100 * vacios, 2), decision, motivo])

base_limpia = original.drop(columns=eliminar).copy()
print(f"{len(base_limpia):,} filas | {len(eliminar)} columnas eliminadas | {len(base_limpia.columns)} conservadas")
reporte = pd.DataFrame(revision_columnas, columns=["Columna", "% vacío", "Decisión", "Motivo"])
display(reporte)

# Guardar el resultado junto al archivo original.
salida_limpia = ruta.with_name("DataLimpia.xlsx")
with pd.ExcelWriter(salida_limpia, engine="openpyxl") as libro:
    base_limpia.to_excel(libro, sheet_name="Productos", index=False)
    reporte.to_excel(libro, sheet_name="Cambios", index=False)
print(f"Base limpia guardada en: {salida_limpia.resolve()}")


5,500 filas | 13 columnas eliminadas | 39 conservadas


,Columna,% vacío,Decisión,Motivo
0,ID_registro,0.00,Conservar,Columna con datos
1,Catalogo_ampliado,0.00,Conservar,Columna con datos
2,Etiquetas,0.00,Eliminar,Contiene enlaces o su encabezado indica enlaces
3,Envio_proveedor,0.07,Conservar,Columna con datos
4,Palabras_clave,1.65,Conservar,Campo protegido del comparador
5,Tiempo_envio,2.35,Conservar,Columna con datos
6,Nombre_producto,0.00,Conservar,Campo protegido del comparador
7,Puntuacion_busqueda,0.00,Conservar,Columna con datos
8,URL_imagen_completa,0.00,Eliminar,Contiene enlaces o su encabezado indica enlaces
9,Etiquetas_acumuladas,79.02,Conservar,Columna con datos


Base limpia guardada en: C:\Users\darro\Mazon\Proyecto segundo parcial\DataLimpia.xlsx


In [14]:
# 4. Agregar variables a la base limpia. Puede ejecutarse después de la celda 1.
from pathlib import Path
from decimal import Decimal
import re
import unicodedata
from html import unescape
import pandas as pd
from IPython.display import display

if "base_limpia" in globals():
    enriquecida = base_limpia.copy()
else:
    enriquecida = pd.read_excel("DataLimpia.xlsx", sheet_name="Productos",
                                dtype={"SKU": str}, keep_default_na=False)

def columna_texto(nombre):
    if nombre not in enriquecida:
        return pd.Series("", index=enriquecida.index, dtype="string")
    return enriquecida[nombre].fillna("").astype("string")

def texto_ficha(valor):
    limpio = unescape(re.sub(r"<[^>]*>", " ", str(valor))).lower()
    return " ".join("".join(c for c in unicodedata.normalize("NFD", limpio)
                            if not unicodedata.combining(c)).split())

# Precios en MXN. Conservar las columnas fuente para poder revisar el cálculo.
def importe_mxn(valor):
    s = re.sub(r"MXN|\$|\s", "", str(valor), flags=re.I)
    if re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", s):
        s = s.replace(",", "")
    elif re.fullmatch(r"\d+,\d{1,2}", s):
        s = s.replace(",", ".")
    n = pd.to_numeric(s, errors="coerce")
    return round(n, 2) if pd.notna(n) and 0 < n < float("inf") else float("nan")

actual = columna_texto("Precio_actual").map(importe_mxn)
previo = columna_texto("Precio_anterior").map(importe_mxn)
enriquecida["Precio_actual_MXN"] = actual
enriquecida["Precio_anterior_MXN"] = previo
diferencia = (previo - actual).round(2)
enriquecida["Descuento"] = diferencia.clip(lower=0)
enriquecida["Descuento_porcentaje"] = (enriquecida["Descuento"] / previo * 100).round(2)
enriquecida["Movimiento_precio"] = "Sin referencia válida"
for condicion, etiqueta in [(diferencia.eq(0), "Sin cambio"), (diferencia.gt(0), "Descuento"),
                             (diferencia.lt(0), "Aumento")]:
    enriquecida.loc[condicion, "Movimiento_precio"] = etiqueta

# Usar las categorías declaradas por la tienda; puede haber varias especialidades.
def categorias_ficha(texto):
    segmentos = [s.strip() for s in texto.split("|")]
    especialidades = []
    for s in segmentos:
        m = re.fullmatch(r"(.+?)\s*[-–]\s*Especialidades\s*[-–]\s*Farmacia", s, re.I)
        if m and m[1].strip() not in especialidades:
            especialidades.append(m[1].strip())
    es_especial = especialidades or any(re.fullmatch(r"Especialidades\s*[-–]\s*Farmacia", s, re.I)
                                         for s in segmentos)
    categoria = "Sin categoría identificada"
    if es_especial:
        categoria = "Especialidades - Farmacia"
    elif any(re.search(r"\bfarmacia\b", s, re.I) for s in segmentos):
        categoria = "Farmacia"
    return "; ".join(especialidades) or "Sin especialidad identificada", categoria

def agregar_campos(nombres, registros):
    enriquecida[nombres] = pd.DataFrame(list(registros), index=enriquecida.index, columns=nombres)

categorias_fuente = columna_texto("Categorias_encontradas") + " | " + columna_texto("Categoria_origen")
agregar_campos(["Especialidad", "Categoria"], categorias_fuente.map(categorias_ficha))

# Lista editable. Registrar solo coincidencias explícitas en los campos indicados.
sustancias = ["Paracetamol", "Ibuprofeno", "Naproxeno", "Diclofenaco", "Omeprazol",
              "Pantoprazol", "Metformina", "Losartán", "Amoxicilina", "Azitromicina",
              "Loratadina", "Cetirizina", "Atorvastatina", "Simvastatina", "Clonazepam",
              "Ácido clavulánico", "Ácido acetilsalicílico", "Magaldrato", "Dimeticona"]
patrones_sustancia = [(s, re.compile(r"(?<![a-z])" + re.escape(texto_ficha(s)) + r"(?![a-z])"))
                      for s in sustancias]

def sustancias_en(texto):
    return " + ".join(s for s, patron in patrones_sustancia if patron.search(texto))

titulos = columna_texto("Nombre_producto").map(texto_ficha)
enriquecida["Sustancia_activa"] = titulos.map(sustancias_en)
texto_apoyo = columna_texto("Descripcion_corta") + " | " + columna_texto("Palabras_clave")
enriquecida["Sustancia_candidata"] = texto_apoyo.map(texto_ficha).map(sustancias_en)

# Reconocer primero razones como 200 mg/5 ml y conservar el texto que originó el cálculo.
num = r"\d+(?:[.,]\d+)?"
unidad = r"(?:mcg|[µμu]g|mg|g|u\.?i\.?|%)"
potencia = re.compile(rf"(?<![\d.,])({num})\s*({unidad})(?![a-z])"
                      rf"(?:\s*/\s*({num})?\s*(ml|g)\b)?", re.I)
def concentracion_ficha(titulo):
    cantidades, unidades, evidencia = [], [], []
    # Una unidad compartida (p. ej. 50/850 mg) necesita interpretación por componente.
    if re.search(rf"{num}\s*/\s*{num}\s*{unidad}(?![a-z])", titulo):
        return "", "", "Revisar la combinación en el título"
    for m in potencia.finditer(titulo):
        valor = Decimal(m[1].replace(",", "."))
        u = m[2].lower().replace(".", "")
        solido = re.search(r"\b(?:tabs?|tabletas?|caps?|capsulas?|comprimidos?)\b", titulo)
        if u == "g" and not m[4] and not solido:
            continue  # Los gramos solos en tubos y frascos suelen indicar contenido neto.
        if u == "g":
            valor, u = valor * 1000, "mg"
        elif u in ("mcg", "µg", "μg", "ug"):
            valor, u = valor / 1000, "mg"
        elif u == "ui":
            u = "UI"
        if m[4]:
            divisor = Decimal((m[3] or "1").replace(",", "."))
            if not divisor:
                return "", "", "Revisar denominador cero"
            valor, u = valor / divisor, u + "/" + m[4]
        cantidades.append(format(valor.normalize(), "f"))
        unidades.append(u)
        evidencia.append(m[0])
    return "; ".join(cantidades), "; ".join(unidades), "; ".join(evidencia)

agregar_campos(["Concentracion", "Unidad_concentracion", "Evidencia_concentracion"],
               titulos.map(concentracion_ficha))

# Unidades físicas declaradas en el envase. No representan una pauta de administración.
formas = {"Tabletas": r"tabletas?|tabs?", "Cápsulas": r"capsulas?|caps?",
          "Comprimidos": r"comprimidos?", "Sobres": r"sobres?", "Ampolletas": r"ampolletas?|ampollas?",
          "Unidades": r"unidades|unidad", "Piezas": r"piezas?|pzas?|pzs?",
          "Grageas": r"grageas?", "Supositorios": r"supositorios?", "Parches": r"parches?"}
tipo = "|".join(formas.values())
conteo = re.compile(rf"(?<![\d.,])(\d+)\s*({tipo})\b")
def presentacion_ficha(titulo):
    halladas = [etiqueta for etiqueta, patron in formas.items()
                if re.search(rf"(?<![a-z])(?:{patron})\b", titulo)]
    piezas = list(conteo.finditer(titulo))
    multiple = re.search(r"\b(?:packs?|kits?|cajas)\b|\+|\d\s*[x×]\s*\d+\s*(?:"+tipo+r")\b", titulo)
    if len(piezas) == 1 and len(halladas) == 1 and not multiple:
        return int(piezas[0][1]), halladas[0]
    return pd.NA, "; ".join(halladas)

agregar_campos(["Unidades_por_envase", "Presentacion"], titulos.map(presentacion_ficha))
enriquecida["Unidades_por_envase"] = enriquecida["Unidades_por_envase"].astype("Int64")
salida_enriquecida = Path("DataEnriquecida.xlsx")
enriquecida.to_excel(salida_enriquecida, sheet_name="Productos", index=False, engine="openpyxl")
print(f"{len(enriquecida):,} filas guardadas en {salida_enriquecida.resolve()}")
display(enriquecida[["SKU", "Nombre_producto", "Descuento", "Descuento_porcentaje", "Especialidad",
                     "Sustancia_activa", "Concentracion", "Unidad_concentracion",
                     "Unidades_por_envase", "Presentacion"]].head(10))


5,500 filas guardadas en C:\Users\darro\Mazon\Proyecto segundo parcial\DataEnriquecida.xlsx


,SKU,Nombre_producto,Descuento,Descuento_porcentaje,Especialidad,Sustancia_activa,Concentracion,Unidad_concentracion,Unidades_por_envase,Presentacion
0,8020030095076,Esoxx One 20 Unidades 10 ml,199.50,30.0,Cardiología; Gastroenterología,,,,20,Unidades
1,7501092793045,Riopan Gel Oral 20 sobres,0.00,0.0,Gastroenterología,,,,20,Sobres
2,7502223709348,Omeprazol 20 mg Oral 30 cápsulas 2 Cajas Marca...,0.00,0.0,Gastroenterología,Omeprazol,20,mg,<NA>,Cápsulas
3,7501088505126,Neuralin Solución Inyectable 2 pz,86.37,37.0,Sin especialidad identificada,,,,2,Piezas
4,7501008443033,Alka Seltzer Alivio del Malestar Estomacal y D...,0.00,0.0,Gastroenterología,,,,12,Tabletas
5,7506472800788,Pinaverio Dimeticona 16 cápsulas Marca del Ahorro,0.00,0.0,Gastroenterología,Dimeticona,,,16,Cápsulas
6,7501328980119,Enterogermina Suspensión de Bacillus Clausii c...,0.00,0.0,Gastroenterología,,,,20,Ampolletas
7,7501298217635,"Dolo-Neurobión Forte DC Diclofenaco Sódico, Co...",0.00,0.0,Sin especialidad identificada,Diclofenaco,,,<NA>,
8,7502223705586,Ibuprofeno 600 mg/10 cápsulas Marca del Ahorro,0.00,0.0,Sin especialidad identificada,Ibuprofeno,600,mg,10,Cápsulas
9,7501318652453,Actron 600 mg con 10 Caps,0.00,0.0,Sin especialidad identificada,,600,mg,10,Cápsulas


In [16]:
# 5. Preparar las variables para Monte Carlo. Ejecutar después de la celda 4.
# También se puede ejecutar por separado si ya existe DataEnriquecida.xlsx.
from pathlib import Path
from decimal import Decimal, InvalidOperation
import re
import unicodedata
import pandas as pd
from IPython.display import display

datos_mc = (enriquecida.copy() if "enriquecida" in globals() else
            pd.read_excel("DataEnriquecida.xlsx", sheet_name="Productos",
                          dtype={"SKU": str, "Concentracion": str}, keep_default_na=False))

def plano_mc(s):
    texto = unicodedata.normalize("NFD", str(s).casefold())
    return " ".join("".join(c for c in texto if not unicodedata.combining(c)).split())

# Clases de referencia para las sustancias de la celda 4; una mención en el título
# es una propuesta de clasificación. Las combinaciones se revisan por producto.
clases_mc = {
    "losartan": "Antihipertensivos (ARA II)", "metformina": "Antidiabéticos (biguanidas)",
    "omeprazol": "Inhibidores de la bomba de protones", "pantoprazol": "Inhibidores de la bomba de protones",
    "paracetamol": "Analgésicos y antipiréticos", "ibuprofeno": "Antiinflamatorios no esteroideos",
    "diclofenaco": "Antiinflamatorios no esteroideos", "naproxeno": "Antiinflamatorios no esteroideos",
    "amoxicilina": "Antibióticos (penicilinas)", "azitromicina": "Antibióticos (macrólidos)",
    "loratadina": "Antihistamínicos", "cetirizina": "Antihistamínicos",
    "atorvastatina": "Hipolipemiantes (estatinas)", "simvastatina": "Hipolipemiantes (estatinas)",
    "clonazepam": "Benzodiazepinas", "magaldrato": "Antiácidos",
}
# Opcional: completar con datos que hayas comprobado en una ficha por SKU.
# Registra Fuente para conservar la referencia. Incluye solo los campos revisados.
# Ejemplo de estructura, para sustituir por el SKU y la ficha reales:
# "SKU_REAL": {"Es_medicamento": 1, "Via_administracion": "Oral",
#              "Tipo_liberacion": "Inmediata", "Tipo_medicamento": "Genérico", "Fuente": "..."}
fichas_revisadas = {}

def etiqueta_mc(texto, reglas):
    coincidencias = [valor for valor, patron in reglas.items() if re.search(patron, texto)]
    return coincidencias[0] if len(coincidencias) == 1 else ""

vias = {"Oral": r"\boral\b", "Sublingual": r"\bsublingual\b", "Oftálmica": r"\boftalmic[ao]\b",
        "Ótica": r"\botic[ao]\b", "Nasal": r"\bnasal\b", "Vaginal": r"\bvaginal\b",
        "Rectal": r"\brectal\b", "Cutánea": r"\b(?:cutane[ao]|topic[ao])\b",
        "Intravenosa": r"\bintravenos[ao]\b", "Intramuscular": r"\bintramuscular\b",
        "Subcutánea": r"\bsubcutane[ao]\b", "Inhalatoria": r"\binhalatori[ao]\b"}
liberaciones = {"Inmediata": r"\bliberacion inmediata\b",
                "Prolongada": r"\bliberacion prolongada\b",
                "Retardada": r"\bliberacion retardada\b",
                "Modificada": r"\bliberacion modificada\b",
                "Entérica": r"\b(?:recubrimiento enterico|gastroresistente)\b"}
titulos_mc = datos_mc.Nombre_producto.map(plano_mc)
activos_mc = datos_mc.Sustancia_activa.fillna("").map(plano_mc)
datos_mc["Clase_terapeutica"] = activos_mc.map(clases_mc).fillna("")
datos_mc["Via_administracion"] = titulos_mc.map(lambda t: etiqueta_mc(t, vias))
datos_mc["Tipo_liberacion"] = titulos_mc.map(lambda t: etiqueta_mc(t, liberaciones))
datos_mc["Tipo_medicamento"] = ""  # Requiere identificación específica; la marca propia es insuficiente.
datos_mc["Es_medicamento"] = pd.Series(pd.NA, index=datos_mc.index, dtype="Int64")
material = titulos_mc.str.match(r"^(?:vendas?|gasas?|guantes?|cubrebocas|termometros?|glucometros?)\b")
material &= ~titulos_mc.str.contains(r"medicad|impregnad") & activos_mc.eq("")
datos_mc.loc[material, "Es_medicamento"] = 0
solidas = {"Tabletas", "Cápsulas", "Comprimidos", "Grageas", "Supositorios", "Óvulos"}
candidato = activos_mc.isin(clases_mc) & datos_mc.Presentacion.isin(solidas)
datos_mc.loc[candidato, "Es_medicamento"] = 1
datos_mc["Fuente_revision"] = ""
editables = {"Clase_terapeutica", "Sustancia_activa", "Concentracion", "Unidad_concentracion",
             "Unidades_por_envase", "Presentacion", "Via_administracion", "Tipo_liberacion",
             "Tipo_medicamento", "Es_medicamento"}
for sku, ficha in fichas_revisadas.items():
    seleccion = datos_mc.SKU.astype(str).eq(str(sku))
    if seleccion.sum() != 1 or not str(ficha.get("Fuente", "")).strip() or set(ficha)-editables-{"Fuente"}:
        raise ValueError(f"Revisa el SKU, los campos y la fuente de la ficha {sku}.")
    for campo, valor in ficha.items():
        datos_mc.loc[seleccion, "Fuente_revision" if campo == "Fuente" else campo] = valor

# Numero_dosis se usa como número de unidades discretas del envase para comparar precios.
# Una tableta es una unidad de referencia; la dosis prescrita podría ser diferente.
unidades = pd.to_numeric(datos_mc.Unidades_por_envase, errors="coerce")
es_med = datos_mc.Es_medicamento.eq(1).fillna(False)
es_discreta = es_med & datos_mc.Presentacion.isin(solidas) & unidades.gt(0) & unidades.mod(1).eq(0)
datos_mc["Numero_dosis"] = unidades.where(es_discreta).astype("Int64")
precios = pd.to_numeric(datos_mc.Precio_actual_MXN, errors="coerce")
datos_mc["Precio_por_dosis"] = precios.where(precios.gt(0)).div(datos_mc.Numero_dosis).round(6)

def numero_clave(valor):
    try:
        n = Decimal(str(valor).strip().replace(",", "."))
        return format(n.normalize(), "f") if n.is_finite() and n > 0 else ""
    except InvalidOperation:
        return ""

# Clave comercial conservadora para sólidos de una sola sustancia y una potencia.
# Requiere vía y liberación conocidas para que dos campos vacíos no generen un par.
def clave_ficha(r):
    if pd.isna(r.Numero_dosis) or pd.isna(r.Es_medicamento) or r.Es_medicamento != 1:
        return ""
    potencia = numero_clave(r.Concentracion)
    identidad = plano_mc(r.Sustancia_activa)
    if not potencia or not identidad or "+" in identidad or not r.Via_administracion or not r.Tipo_liberacion:
        return ""
    if str(r.Unidad_concentracion).strip() not in {"mg", "UI"}:
        return ""
    if not r.Fuente_revision:
        nombre = plano_mc(r.Nombre_producto)
        if not nombre.startswith(identidad) or re.search(r"[/+]|\b(?:duo|compuesto|con cafeina)\b", nombre):
            return ""
    partes = [identidad, potencia + str(r.Unidad_concentracion), str(int(r.Numero_dosis)),
              r.Presentacion, r.Via_administracion, r.Tipo_liberacion]
    return "_".join(plano_mc(p).upper().replace(" ", "-") for p in partes)

datos_mc["Clave_equivalencia"] = [clave_ficha(r) for r in datos_mc.itertuples(index=False)]
sku_mc = datos_mc.SKU.fillna("").astype(str).str.strip()
aptos = datos_mc.Clave_equivalencia.ne("") & datos_mc.Precio_por_dosis.notna() & sku_mc.ne("") & ~sku_mc.duplicated(False)
pares = datos_mc.loc[aptos].groupby("Clave_equivalencia").SKU.transform("nunique")
datos_mc["Es_comparable"] = 0
datos_mc.loc[pares.index[pares.ge(2)], "Es_comparable"] = 1
datos_mc["Estado_validacion"] = "Pendiente"
datos_mc.loc[datos_mc.Es_medicamento.eq(0).fillna(False), "Estado_validacion"] = "Fuera de alcance por clasificación"
datos_mc.loc[datos_mc.Clave_equivalencia.ne(""), "Estado_validacion"] = "Clave generada; revisar ficha"
datos_mc.loc[datos_mc.Fuente_revision.ne(""), "Estado_validacion"] = "Con revisión documentada"

columnas_mc = ["Categoria", "Especialidad", "Clase_terapeutica", "Sustancia_activa", "Concentracion",
               "Unidad_concentracion", "Numero_dosis", "Presentacion", "Via_administracion",
               "Tipo_liberacion", "Tipo_medicamento", "Descuento", "Descuento_porcentaje",
               "Precio_por_dosis", "Clave_equivalencia", "Es_medicamento", "Es_comparable", "Estado_validacion"]
salida_mc = Path("Base_MonteCarlo.xlsx")
datos_mc.to_excel(salida_mc, columns=columnas_mc, sheet_name="Productos", index=False, engine="openpyxl")
print(f"{len(datos_mc):,} filas | Con precio por unidad de referencia: {datos_mc.Precio_por_dosis.notna().sum():,}"
      f" | Comparables con la clave completa: {datos_mc.Es_comparable.sum():,}")
print(f"Base para simulación: {salida_mc.resolve()}")
display(datos_mc[["SKU", "Nombre_producto"] + columnas_mc].head(10))


5,500 filas | Con precio por unidad de referencia: 137 | Comparables con la clave completa: 0
Base para simulación: C:\Users\darro\Mazon\Proyecto segundo parcial\Base_MonteCarlo.xlsx


,SKU,Nombre_producto,Categoria,Especialidad,Clase_terapeutica,Sustancia_activa,Concentracion,Unidad_concentracion,Numero_dosis,Presentacion,Via_administracion,Tipo_liberacion,Tipo_medicamento,Descuento,Descuento_porcentaje,Precio_por_dosis,Clave_equivalencia,Es_medicamento,Es_comparable,Estado_validacion
0,8020030095076,Esoxx One 20 Unidades 10 ml,Especialidades - Farmacia,Cardiología; Gastroenterología,,,,,<NA>,Unidades,,,,199.50,30.0,<NA>,,<NA>,0,Pendiente
1,7501092793045,Riopan Gel Oral 20 sobres,Especialidades - Farmacia,Gastroenterología,,,,,<NA>,Sobres,Oral,,,0.00,0.0,<NA>,,<NA>,0,Pendiente
2,7502223709348,Omeprazol 20 mg Oral 30 cápsulas 2 Cajas Marca...,Especialidades - Farmacia,Gastroenterología,Inhibidores de la bomba de protones,Omeprazol,20,mg,<NA>,Cápsulas,Oral,,,0.00,0.0,<NA>,,1,0,Pendiente
3,7501088505126,Neuralin Solución Inyectable 2 pz,Especialidades - Farmacia,Sin especialidad identificada,,,,,<NA>,Piezas,,,,86.37,37.0,<NA>,,<NA>,0,Pendiente
4,7501008443033,Alka Seltzer Alivio del Malestar Estomacal y D...,Especialidades - Farmacia,Gastroenterología,,,,,<NA>,Tabletas,,,,0.00,0.0,<NA>,,<NA>,0,Pendiente
5,7506472800788,Pinaverio Dimeticona 16 cápsulas Marca del Ahorro,Especialidades - Farmacia,Gastroenterología,,Dimeticona,,,<NA>,Cápsulas,,,,0.00,0.0,<NA>,,<NA>,0,Pendiente
6,7501328980119,Enterogermina Suspensión de Bacillus Clausii c...,Especialidades - Farmacia,Gastroenterología,,,,,<NA>,Ampolletas,,,,0.00,0.0,<NA>,,<NA>,0,Pendiente
7,7501298217635,"Dolo-Neurobión Forte DC Diclofenaco Sódico, Co...",Especialidades - Farmacia,Sin especialidad identificada,Antiinflamatorios no esteroideos,Diclofenaco,,,<NA>,,,,,0.00,0.0,<NA>,,<NA>,0,Pendiente
8,7502223705586,Ibuprofeno 600 mg/10 cápsulas Marca del Ahorro,Especialidades - Farmacia,Sin especialidad identificada,Antiinflamatorios no esteroideos,Ibuprofeno,600,mg,10,Cápsulas,,,,0.00,0.0,8.5,,1,0,Pendiente
9,7501318652453,Actron 600 mg con 10 Caps,Especialidades - Farmacia,Sin especialidad identificada,,,600,mg,<NA>,Cápsulas,,,,0.00,0.0,<NA>,,<NA>,0,Pendiente


In [13]:
# 2. Agrupar por palabras clave. Edita este diccionario y ejecuta otra vez las celdas 2 y 3.
# A la izquierda va el grupo; a la derecha, los términos que lo activan.
catalogo = {
    # Usos y síntomas mencionados en la ficha
    "Reflujo": ["reflujo", "ERGE"],
    "Acidez": ["acidez", "hiperacidez", "agruras", "pirosis"],
    "Gastritis": ["gastritis"],
    "Úlceras digestivas": ["úlcera gástrica", "úlceras gástricas", "úlcera duodenal", "úlcera péptica"],
    "Indigestión": ["indigestión", "dispepsia"],
    "Gases intestinales": ["flatulencia", "flatulencias", "gases intestinales", "meteorismo"],
    "Estreñimiento": ["estreñimiento", "laxante", "laxantes"],
    "Diarrea": ["diarrea", "antidiarreico", "antidiarreicos"],
    "Náuseas y vómito": ["náusea", "náuseas", "vómito", "vómitos", "antiemético"],
    "Dolor de cabeza": ["dolor de cabeza", "cefalea", "cefaleas", "migraña"],
    "Dolor muscular": ["dolor muscular", "dolores musculares", "mialgia"],
    "Dolor dental": ["dolor dental", "dolor de muelas", "dolor de dientes"],
    "Dolor menstrual": ["dolor menstrual", "cólico menstrual", "cólicos menstruales", "dismenorrea"],
    "Fiebre": ["fiebre", "antipirético"],
    "Alergias": ["alergia", "alergias", "antihistamínico", "rinitis alérgica"],
    "Tos": ["tos", "antitusivo"],
    "Gripa y resfriado": ["gripa", "gripe", "resfriado", "resfriados", "antigripal"],
    "Congestión nasal": ["congestión nasal", "descongestionante nasal"],
    "Dolor de garganta": ["dolor de garganta", "irritación de garganta"],
    "Asma": ["asma", "antiasmático"],
    "Hipertensión arterial": ["hipertensión arterial", "presión arterial alta", "antihipertensivo"],
    "Diabetes": ["diabetes", "antidiabético"],
    "Colesterol y triglicéridos": ["colesterol", "triglicéridos", "hipercolesterolemia", "dislipidemia"],
    "Artritis y artrosis": ["artritis", "artrosis", "osteoartritis"],
    "Infección urinaria": ["infección urinaria", "infecciones urinarias", "cistitis", "infección de vías urinarias"],
    "Hongos y micosis": ["micosis", "antimicótico", "antifúngico", "pie de atleta", "candidiasis"],
    "Acné": ["acné"],
    "Dermatitis": ["dermatitis", "eczema", "eccema"],
    "Heridas": ["herida", "heridas", "cicatrización"],
    "Quemaduras": ["quemadura", "quemaduras"],
    "Ojo seco": ["ojo seco", "ojos secos", "sequedad ocular", "lágrimas artificiales"],
    "Ansiedad": ["ansiedad", "ansiolítico"],
    "Depresión": ["depresión", "antidepresivo"],
    "Insomnio": ["insomnio"],
    "Epilepsia y convulsiones": ["epilepsia", "convulsiones", "anticonvulsivo"],
    "Anticoncepción": ["anticoncepción", "anticonceptivo", "anticonceptivos", "anticonceptiva"],
    "Anemia": ["anemia"],
    "Osteoporosis": ["osteoporosis"],
    "Hipotiroidismo": ["hipotiroidismo"],
    "Vértigo y mareos": ["vértigo", "mareo", "mareos"],
    "Hemorroides": ["hemorroides"],
    "Parkinson": ["Parkinson"],
    "Menopausia": ["menopausia"],
    "Psoriasis": ["psoriasis"],
    "Incontinencia": ["incontinencia"],
    "Protección solar": ["protector solar", "protección solar", "fotoprotector"],
    "Tabaquismo": ["tabaquismo", "dejar de fumar"],
    "Higiene bucal": ["higiene bucal", "enjuague bucal", "gingivitis", "caries"],
    "Control de peso": ["control de peso", "obesidad", "sobrepeso"],

    # Material de curación, dispositivos y otros productos
    "Vendas": ["venda", "vendas"],
    "Gasas": ["gasa", "gasas"],
    "Apósitos": ["apósito", "apósitos"],
    "Cintas médicas": ["cinta micropore", "cinta quirúrgica", "tela adhesiva", "micropore"],
    "Curitas": ["curitas", "venditas adhesivas"],
    "Algodón": ["algodón"],
    "Guantes": ["guante", "guantes"],
    "Jeringas": ["jeringa", "jeringas"],
    "Agujas": ["aguja", "agujas"],
    "Lancetas": ["lanceta", "lancetas"],
    "Tiras reactivas": ["tira reactiva", "tiras reactivas"],
    "Glucómetros": ["glucómetro", "glucómetros", "medidor de glucosa"],
    "Termómetros": ["termómetro", "termómetros"],
    "Oxímetros": ["oxímetro", "oxímetros"],
    "Medidores de presión": ["baumanómetro", "baumanómetros", "tensiómetro", "monitor de presión"],
    "Nebulizadores": ["nebulizador", "nebulizadores"],
    "Cubrebocas": ["cubrebocas", "mascarilla quirúrgica"],
    "Férulas": ["férula", "férulas"],
    "Plantillas para calzado": ["plantilla", "plantillas"],
    "Tobilleras": ["tobillera", "tobilleras"],
    "Muñequeras": ["muñequera", "muñequeras"],
    "Pruebas de embarazo": ["prueba de embarazo", "pruebas de embarazo"],
    "Preservativos": ["condón", "condones", "preservativo", "preservativos"],
    "Lubricantes íntimos": ["lubricante íntimo", "lubricante personal", "gel lubricante"],
    "Probióticos": ["probiótico", "probióticos"],
    "Electrolitos": ["electrolito", "electrolitos", "suero oral"],
    "Multivitamínicos": ["multivitamínico", "multivitamínicos"],
    "Vitamina C": ["vitamina C", "ácido ascórbico"],
    "Vitamina D": ["vitamina D", "vitamina D3", "colecalciferol"],
    "Complejo B": ["complejo B", "complejo vitamínico B"],
    "Ácido fólico": ["ácido fólico"],

    # Principios activos y nombres de búsqueda
    "Aspirina": ["aspirina", "ácido acetilsalicílico"],
    "Ibuprofeno": ["ibuprofeno", "Advil", "Motrin"],
    "Paracetamol": ["paracetamol", "acetaminofén"],
    "Naproxeno": ["naproxeno"],
    "Diclofenaco": ["diclofenaco"],
    "Ketorolaco": ["ketorolaco"],
    "Metamizol": ["metamizol", "dipirona"],
    "Meloxicam": ["meloxicam"],
    "Celecoxib": ["celecoxib"],
    "Tramadol": ["tramadol"],
    "Omeprazol": ["omeprazol"],
    "Pantoprazol": ["pantoprazol"],
    "Esomeprazol": ["esomeprazol"],
    "Lansoprazol": ["lansoprazol"],
    "Dexlansoprazol": ["dexlansoprazol"],
    "Famotidina": ["famotidina"],
    "Magaldrato": ["magaldrato"],
    "Simeticona": ["simeticona"],
    "Loperamida": ["loperamida"],
    "Metoclopramida": ["metoclopramida"],
    "Ondansetrón": ["ondansetrón"],
    "Pinaverio": ["pinaverio", "bromuro de pinaverio"],
    "Trimebutina": ["trimebutina"],
    "Loratadina": ["loratadina"],
    "Desloratadina": ["desloratadina"],
    "Cetirizina": ["cetirizina"],
    "Levocetirizina": ["levocetirizina"],
    "Fexofenadina": ["fexofenadina"],
    "Montelukast": ["montelukast"],
    "Salbutamol": ["salbutamol"],
    "Budesonida": ["budesonida"],
    "Ambroxol": ["ambroxol"],
    "Acetilcisteína": ["acetilcisteína"],
    "Dextrometorfano": ["dextrometorfano"],
    "Amoxicilina": ["amoxicilina"],
    "Azitromicina": ["azitromicina"],
    "Claritromicina": ["claritromicina"],
    "Cefalexina": ["cefalexina"],
    "Ceftriaxona": ["ceftriaxona"],
    "Ciprofloxacino": ["ciprofloxacino"],
    "Levofloxacino": ["levofloxacino"],
    "Clindamicina": ["clindamicina"],
    "Metronidazol": ["metronidazol"],
    "Fluconazol": ["fluconazol"],
    "Clotrimazol": ["clotrimazol"],
    "Ketoconazol": ["ketoconazol"],
    "Terbinafina": ["terbinafina"],
    "Aciclovir": ["aciclovir"],
    "Metformina": ["metformina"],
    "Glimepirida": ["glimepirida"],
    "Sitagliptina": ["sitagliptina"],
    "Empagliflozina": ["empagliflozina"],
    "Insulina": ["insulina", "insulinas"],
    "Levotiroxina": ["levotiroxina"],
    "Losartán": ["losartán"],
    "Telmisartán": ["telmisartán"],
    "Irbesartán": ["irbesartán"],
    "Valsartán": ["valsartán"],
    "Enalapril": ["enalapril"],
    "Amlodipino": ["amlodipino"],
    "Metoprolol": ["metoprolol"],
    "Bisoprolol": ["bisoprolol"],
    "Furosemida": ["furosemida"],
    "Espironolactona": ["espironolactona"],
    "Atorvastatina": ["atorvastatina"],
    "Rosuvastatina": ["rosuvastatina"],
    "Clopidogrel": ["clopidogrel"],
    "Apixabán": ["apixabán"],
    "Rivaroxabán": ["rivaroxabán"],
    "Prednisona": ["prednisona"],
    "Dexametasona": ["dexametasona"],
    "Betametasona": ["betametasona"],
    "Hidrocortisona": ["hidrocortisona"],
    "Pregabalina": ["pregabalina"],
    "Gabapentina": ["gabapentina"],
    "Lamotrigina": ["lamotrigina"],
    "Levetiracetam": ["levetiracetam"],
    "Sertralina": ["sertralina"],
    "Escitalopram": ["escitalopram"],
    "Fluoxetina": ["fluoxetina"],
    "Quetiapina": ["quetiapina"],
    "Tadalafilo": ["tadalafilo", "tadalafil"],
    "Sildenafilo": ["sildenafilo", "sildenafil"],

}

# Familias comerciales y especialidades. Estas categorías comparan desembolso por envase.
catalogo.update({
    "Cardiología": ["cardiología", "cardiovascular", "cardiovasculares", "cardiaco", "cardiaca"],
    "Gastroenterología": ["gastroenterología", "gastrointestinal", "digestivo", "digestiva"],
    "Dermatología": ["dermatología", "dermatológico", "cuidado de la piel"],
    "Neurología": ["neurología", "neurológico", "sistema nervioso"],
    "Ginecología": ["ginecología", "ginecológico", "ginecológica"],
    "Oftalmología": ["oftalmología", "oftálmico", "oftálmica", "ocular"],
    "Oncología": ["oncología", "quimioterapia", "oncológico"],
    "Urología": ["urología", "urológico"],
    "Endocrinología": ["endocrinología", "endocrino", "endocrinológico"],
    "Infectología": ["infectología", "antibiótico", "antibióticos", "antibacteriano"],
    "Traumatología y ortopedia": ["ortopedia", "ortopédico", "ortopédica", "traumatología"],
    "Hematología": ["hematología", "hematológico"],
    "Salud mental": ["salud mental", "psiquiatría"],
    "Material de curación": ["curación", "primeros auxilios"],
    "Suplementos": ["suplemento", "suplementos", "suplemento alimenticio"],
    "Hipertensión": ["hipertensión", "hipertensivo", "antihipertensivos"],
    "Antiinflamatorios": ["antiinflamatorio", "antiinflamatorios", "antiinflamatoria"],
    "Analgésicos": ["analgésico", "analgésicos", "analgésica"],
    "Hormonas": ["hormona", "hormonas", "hormonal"],
    "Anticoagulantes": ["anticoagulante", "anticoagulantes"],
    "Hidratación": ["rehidratación", "hidratación", "hidratante"],
    "Antisépticos": ["antiséptico", "antiséptica", "desinfectante"],
    "Protección íntima": ["toalla sanitaria", "protectores diarios", "tampones"],
    "Cuidado de pies": ["cuidado de los pies", "pies", "pie de atleta"],
    "Accesorios íntimos": ["vibrador", "masajeador íntimo", "estimulador"],
})
# Presentación declarada: permite comparar precios de etiqueta dentro de un formato.
formatos = {
    "Tabletas": ["tabletas", "tableta", "tabs", "tab", "comprimido", "comprimidos"],
    "Cápsulas": ["cápsulas", "cápsula", "caps", "cap"],
    "Grageas": ["gragea", "grageas"],
    "Soluciones": ["solución", "soluciones"],
    "Suspensiones": ["suspensión", "suspensiones"],
    "Jarabes": ["jarabe", "jarabes"],
    "Gotas": ["gotas", "gotero"],
    "Inyectables": ["inyectable", "inyectables", "inyección", "ampolleta", "ampolletas", "ampolla", "ampollas"],
    "Cremas": ["crema", "cremas"],
    "Ungüentos": ["ungüento", "ungüentos", "pomada", "pomadas"],
    "Geles": ["gel", "geles"],
    "Aerosoles": ["aerosol", "aerosoles", "spray"],
    "Sobres": ["sobres", "sobre monodosis"],
    "Parches": ["parche", "parches"],
    "Óvulos": ["óvulo", "óvulos"],
    "Supositorios": ["supositorio", "supositorios"],
    "Polvos": ["polvo", "polvos", "granulado"],
    "Lociones": ["loción", "lociones"],
    "Champús": ["shampoo", "champú", "champús"],
    "Pastillas": ["pastilla", "pastillas"],
}
catalogo.update({"Presentación · " + grupo: palabras for grupo, palabras in formatos.items()})

# Líneas de producto repetidas en el catálogo original. Cada término crea su propio grupo.
lineas_producto = (
    "abasaglar acciogen accu aceler aclarium actron acutears acxion adaferin adarex "
    "adel aderogyl adial aerochamber aeroespaciador afrin agreless agrifen agua ahlib "
    "aittaret akabar akineton albothyl alcohol aldactone aldomet alegoria alevian alfa "
    "alflorex algidol algitrin alin aliren alka allegra allitriple alliviax almax "
    "almetec almohadillas altiorem aluminio alzam amably amal amantadina amaryl amikacina "
    "amizt ammens amoebriz amoxibron amoxiclav amoxil ampicilina ampliron anaferon analgen "
    "anapsique anara andantol angiotrofin antiacidos antiflu anuar anubis anzanera aplicadores "
    "aprovasc aprovel arahkor arfla argentafil arluy arnaltem arquera artelac asenlix "
    "atacand atarax atemperator atozet augmentin avalide avanzaten avapena aviant avirena "
    "azibiot azitrocin bactocin bactrim bactroban bafexal balbit bandas bandita bartilixima "
    "batensiar baycuten bebida belara benadryl bencidamina bencilpenicilina benedorm beniflant benzal "
    "benzonatato bepanthen berocca betnovate bicarbonato bicartial biconcor bifebral bilina binotal "
    "bioderma biogaia biolaif biometrix biotrefon bisenka bisolfam bisolvon blaxitec blistex "
    "blitsons blodivit blopress bojidar bolentax bonadoxina boost botiquin bredelin brilinta "
    "bromhexina broncho broncolin broxol bullenza buscapina butilhioscina bykypad cabergolina caduet "
    "cafiaspirina caladryl calcort caltrate candiphen canescalm canesten captopril captral carbazina "
    "cardinit cardispan cardomicin cardura carnotprim carvedilol cataflam caudaline cecilet cedax "
    "cefaxona cefixima cefotaxima ceftrex cefuracet cefuroxima celestamine celestone centrum cerave "
    "cervilan cetolan chipten cialis cibux cicadona cicloferon ciproflox ciproxina ciruelax "
    "cisaprida citizen clarinex clarityne clauter clavulin clearblue clearmicin clendix clexane "
    "clobexpro clonazepam clonixinato cloramfeni clorfenamina colageno collarin colpunov combesteral complement "
    "compresa conagrad conair conazol concor conforiar contour controlip cormocor corpotasin "
    "coversam cozaar creon crestor criam crisvi curas cureband dabex daflon "
    "dafloxen dailies daivobet dalacin dalux dalvear danzen daonsa dapagliflozina daterisina "
    "davalox daxon debromu decorex deflamox deflazacort deflox degregan denvar deples "
    "depo deprectal derfina derluma derman dermatovate dersupril desenex desenfriol desowen "
    "despamen desvenlafaxina devinar devixdol dexabion dexgstrol dexivant dextion diasporal dicetel "
    "dicloxacilina dicynone difenhidramina difenidol dilacoran dilarmine dilatrend dimacol dimefor dimegan "
    "dimenhidrinato dimoflax dinexa diosmina diovan diprosone diprospan dislep dismedox dispositivo "
    "divelgel dolac dolne dolo dolotandax dominion donodol dorixina dorso dosier "
    "dostein dostinex doxiciclina dramamine drence dropropizina dubila duloxetina dunoxsol duoalmetec "
    "duoclindox duoxin durater duspatalin dutina eborix ectaprim edarbi effezel elantan "
    "elatec eldopaque eldoquin electrolife electrolit elicuis elidel emla emulsion enaladil "
    "enjuague ensure enterex enterogermina entresto epamin epuris equivac eritromicina eskapar "
    "esmisen espacil espaven espidorm esteclin estericide esteriflu estetol esthipia estomaquil "
    "estrogenos eternal etoricoxib etral eturion eucaliptine eucerin eufenil eutirox evastel "
    "evemglea everest evipress evocs exbuten excedrin exforge exgerm exotib exotrima "
    "eyestil ezetimiba ezetrol facicam fanter farmapram farmaxetina febratic febrax felodipino "
    "femisan femoston ferranina ferricol ficonax fijador firac firsito fitoestimulina flagenase "
    "flagyl flanax flevox flonorm floratil flos flumil fluocinolona fluoning fluoxac "
    "fluxedan folivital forxiga fosfocil fosfomicina fossin fotexina foundayz freestyle fucidin "
    "furacin gabantin gabirol galactus galvus gamo ganglioside gangotri gantena gapridol "
    "garamicina gelan genoprazol gentamicina gentil geslutin gestageno ginorelle glanrotir glimetal "
    "glioten glitacar glucerna glucophage glucosa glucovance glutapak gotinal graneodin granudoxy "
    "gremiltal guaifenesina haloperil heally hedera hemosin hepa herklin hesven hidrasec "
    "higroton hioscina hisopos histiacil histofil humalo humalog humylub hyalo hydrashot "
    "hydrasor hyzaar iberoflora iberogast idaptan ifor iliadin ilimit ilosone iltux "
    "iltuxam immu incresina indaflex inderalici indocid indometacina inhepar inhibitron insogen "
    "inspra insulex intimina invictus invokana iodopovidona iprikene isdin isodine isoprinosine "
    "isorbid isotretinoina isox italdermol itraconazol itravil ivexterm jaloma janumet januvia "
    "jard jardianz jarsix kabir kaleidon kaomycin karet karolus karzatec keflex "
    "ketoprofeno kinestase kirruz kisika kitoscell klaricid klimi klodex kombiglyze koptin "
    "kriadex labixten lactacyd lacteol lactipan lactulax lamisil lamobrigan lantus laser "
    "lasix lasmaskin latasina laxoya lefloxin lelo leneda leptopsique lerk leukoplast "
    "levante levigrix levitra levonorgestrel liberan liberdux libertrim libonide lidocaina lievricin "
    "limagal linazic lincocin lincomicina lipidil lipitor lirselude lixiana lobi lobivon "
    "loceryl locoid lodarta lodestar lomecan lometopan lomexin lomotil lonol lopid "
    "lopresor loterex lotrimin loventrax loxcell loxonin lubricante lunarium luvik lysomucil "
    "macmirror macrozit madecassol madrilcan makzim manzanilla masennus maxifort maxopress medibutin "
    "medicor medipiox mefiros megion meibi mejoral meladinina melox merthiolate mertodol "
    "metamucil metcolgrass meteospasmyl meticel meticortelone meticorten mexsana micardis miccil microbiot "
    "microgynon midenar miflonide minipres minirin minitran minocin minodiab misdapre mitzoratta "
    "mometasona monitor monocorat montaclar morebon motilium mounjaro movuren moxazyd mucosolvan "
    "mucovibrol multiblue multimetro mutel mutirot muvixden mycelvan nabian nafazolina nailex "
    "nari nartex nasalub nativa naturalag naturelab naxifelar nazil neat neilmed "
    "neocholal neomicina neomicol neoseverin neositamet neotrex neparvis nepro nesajar neugeron "
    "neupax neuralin neurobion neutek nevrika nexcare nexium next nexus nifedipino "
    "nikzon niquitin nitazoxanida nivelare nizoral norfenon normex norutec norvas novolin "
    "novorapid nucleo nulytely nutribio obeclox ogmatin olmesartan olorex olormax omega "
    "omron omuro onetouch onglyza openvas optive oranor oratane orlistat oroxadin "
    "ortopsique otedram ovestin oxal oxcarbazepina oxcer oxeladina oxital ozempic pads "
    "pamezone panclasa pantozol paramix parkpex pasta patector patrex pearls pedialyte "
    "pediasure peine pemix penamox penbritin penprocilina pentasa pepsane periamid periplum "
    "pharmafet pharmaton phlebodia picot pilefhid pilovait pioglitazona piracetam piroxicam pirquet "
    "pisalak pisalpra pisatine pisaurit pisavit plastipak plavix playboy plemtum plenacor "
    "plexus plidan polixin pontiride posipen postday poverful pradaxar pramigel pranosine "
    "prasucor pravastatina prazolan predial prednisolona pregnus premarin presevativos presone prikul "
    "primogyn prindex probador probiolog procoralan procto profenid prolub promotion protec "
    "proteflor provera prudence psyllium pulmicort quinfamida raas radiance ramipril rapix "
    "rebiot rebless recoveron reddy redevant redoxon redustat refresh regulact renu "
    "repafet resotrans respicil retacnyl retin reumophan rexona ricola rimastine rinofren "
    "riopan risperidona robotek rocainol rogastril rolet rosvaden rovartal rulle rupilip "
    "rybelsus ryzodeg saari sabro salofalk salpifar samyr saridon scabisan sedalmerck "
    "sedasiva segmir seloken seltaferon seltz senex senociclin senokot senosiain sensibit "
    "sensidex septacin sermion seronex sertex sico silka silkamedic simplex singari "
    "sinomarin sinuberase siptatin siptatinmet solievo soliqua solutina somazina sonda soyaloid "
    "spasmopriv splenda stadium staquis sterimar sucrato suerox sufrexal sukrol sunderin "
    "supacid supradol synalar syncol synthroid syptime systane tabalon tabcin taedallis "
    "tafil tafirol tafitram talco tamiram tamsulosina tapazol tarka taxus tebonin "
    "tecta tela telarteq temerit temitev tempra temprafen tenoretic tenormin teofilina "
    "teolong tesalon tetralisal tevetenz theraflu thevier thoreva tiaminal tiazopir ting "
    "tirevin tobramicina topifort topron toujeo tradea traderma tradol tratamiento trayenta "
    "trazidex treda tresiba tressvin trevissage trexen trezete tribedoce triderm trifamox "
    "trimetoprima trinomia triplixam tritace tritazide trivas triyotex troferit trojan tronium "
    "trulicity tubo tukeli tukol tumenil tums tusigen tylenol tylex ulcoderma "
    "ulkro ulsen umckaloabo unamol unasal unival urominot ursofalk utrogestan vagisense "
    "vagitrol valextra valmetrol valproato valprosid valvulan vantal variton vasculflow vaso "
    "vastionin vedipal velian velozia venastat venditas ventolin vermox verquvo vertisal "
    "vessel viagra vialibram vibramicina vick vidipep vildagliptina vilona vilopre vilzermet "
    "virazide virlix vitacilina vivioptal vivitar voit voltaren vontrol votripax vytorin "
    "wegovy weserix wild wise woman xaraban xarelto xavi xeletec xerenex "
    "xigduo xiglif xuceed xultophy xuzal yasmin yogi yoogil zaldiar zanidip "
    "zapex zathelo zedesen zentel zerpyco zestoretic zestril zient zinnat zinolox "
    "zintrepid zofran zoltum zonaker zovirax zuum zydilo zymir zyplo zyrtec "
    "zystiq zyxem "
).split()
catalogo.update({"Línea · " + linea.capitalize(): [linea] for linea in lineas_producto})

from html import unescape
import unicodedata

def limpiar_texto(valor):
    return " ".join(re.sub(r"<[^>]*>", " ", unescape(str(valor))).split())

def normalizar(valor):
    texto = unicodedata.normalize("NFD", limpiar_texto(valor).lower())
    return "".join(c for c in texto if not unicodedata.combining(c))

base = base_limpia.copy()
base["nombre_busqueda"] = base["Nombre_producto"].map(normalizar)
base["descripcion_busqueda"] = base["Descripcion_corta"].map(normalizar)

# Un grupo puede coincidir en las dos fuentes. Se acredita solo una vez por fila:
# primero la descripción y, para ese grupo, después las claves de tienda.
# El nombre solo se emplea más adelante para interpretar las cantidades del envase.
base["claves_busqueda"] = base["Palabras_clave"].map(normalizar)
def tokens(s):
    # Separa también formatos unidos, como 30tabs o 40mg.
    return re.findall(r"[^\W\d_]+|\d+", s, re.UNICODE)

# Índice de frases del diccionario. Buscar por primer término evita recorrer
# todas las líneas comerciales por cada producto.
from collections import defaultdict
indice = defaultdict(list)
for grupo, palabras in catalogo.items():
    for palabra in palabras:
        frase = tuple(tokens(normalizar(palabra)))
        if frase:
            indice[frase[0]].append((frase, grupo, palabra))

def coincidencias(texto):
    lista = tokens(texto)
    grupos = {}
    for i, token in enumerate(lista):
        for frase, grupo, palabra in indice.get(token, ()):
            if tuple(lista[i:i+len(frase)]) == frase:
                grupos.setdefault(grupo, palabra)
    return grupos

asignaciones = []
for fila, (desc, claves) in enumerate(zip(base.descripcion_busqueda, base.claves_busqueda)):
    primero = coincidencias(desc)
    segundo = coincidencias(claves)
    for grupo in primero.keys() | segundo.keys():
        asignaciones.append((fila, grupo, primero.get(grupo, ""), segundo.get(grupo,""),
                             "Descripción corta" if grupo in primero else "Palabras clave"))
    if not primero and not segundo:
        asignaciones.append((fila, "Sin clasificar", "", "", "Sin coincidencia"))
etiquetas = pd.DataFrame(asignaciones, columns=["Fila","Grupo","Clave_descripcion","Clave_tienda","Fuente_asignacion"])
columnas = ["SKU","Nombre_producto","Descripcion_corta","Palabras_clave","Precio_actual",
            "nombre_busqueda","descripcion_busqueda"] + [c for c in ["Marca","URL_producto"] if c in base]
agrupados = etiquetas.merge(base[columnas], left_on="Fila", right_index=True, validate="many_to_one")
agrupados = agrupados.sort_values(["Grupo","Fila"]).reset_index(drop=True)
display(agrupados.groupby(["Grupo","Fuente_asignacion"]).size().unstack(fill_value=0))

# Exportar un libro por grupo del diccionario, incluidos los grupos con una sola coincidencia.
# Cada ejecución guarda su propio lote para que puedas editar el catálogo y comparar versiones.
from datetime import datetime
from tempfile import TemporaryDirectory

carpeta_subgrupos = ruta.parent / "subgrupos"
carpeta_subgrupos.mkdir(exist_ok=True)
columnas_excel = ["Grupo", "SKU", "Nombre_producto", "Precio_actual",
                  "Fuente_asignacion", "Clave_descripcion", "Clave_tienda",
                  "Descripcion_corta", "Palabras_clave"]
columnas_excel += [c for c in ["Marca", "URL_producto"] if c in agrupados]
anchos = [34, 20, 70, 16, 23, 25, 25, 80, 65] + [26 if c == "Marca" else 65 for c in columnas_excel[9:]]
nombres_usados = set()

def nombre_excel(grupo):
    limpio = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "-", grupo).strip(" .")[:95].rstrip(" .") or "Grupo"
    if re.fullmatch(r"CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9]", limpio.split(".")[0], re.I):
        limpio = "_" + limpio
    nombre = limpio
    contador = 1
    while nombre.casefold() in nombres_usados:
        contador += 1
        nombre = f"{limpio}_{contador}"
    nombres_usados.add(nombre.casefold())
    return nombre + ".xlsx"

# Publicar el lote completo únicamente cuando se hayan escrito todos sus archivos.
with TemporaryDirectory(dir=carpeta_subgrupos) as temporal:
    lote = Path(temporal) / "libros"
    lote.mkdir()
    grupos_exportados = 0
    for grupo, productos in agrupados.groupby("Grupo", sort=True):
        destino = lote / nombre_excel(grupo)
        with pd.ExcelWriter(destino, engine="openpyxl") as escritor:
            productos[columnas_excel].to_excel(escritor, sheet_name="Productos", index=False)
            hoja = escritor.sheets["Productos"]
            hoja.freeze_panes = "C2"
            hoja.auto_filter.ref = hoja.dimensions
            for celda, ancho in zip(hoja[1], anchos):
                hoja.column_dimensions[celda.column_letter].width = ancho
        grupos_exportados += 1
    carpeta_resultados = carpeta_subgrupos / datetime.now().strftime("%Y-%m-%d_%H-%M-%S_%f")
    lote.rename(carpeta_resultados)
print(f"{grupos_exportados:,} archivos Excel guardados en {carpeta_resultados.resolve()}")


Fuente_asignacion,Descripción corta,Palabras clave,Sin coincidencia
Grupo,,,
Accesorios íntimos,22,7,0
Acetilcisteína,1,7,0
Aciclovir,6,11,0
Acidez,46,50,0
Acné,31,13,0
...,...,...,...
Vitamina C,43,20,0
Vitamina D,42,18,0
Vértigo y mareos,10,6,0


1,142 archivos Excel guardados en C:\Users\darro\Mazon\Proyecto segundo parcial\subgrupos\2026-11-18_23-29-43_956232


In [12]:
# 3. Comparar precios dentro de cada grupo.
# El mínimo por envase mide desembolso. El mínimo unitario exige una presentación comparable.
def presentacion(nombre, descripcion):
    numero = r"(\d+(?:[.,]\d+)?)"
    def valor(s):
        return float(s.replace(",", "."))
    if re.search(r"\b(?:packs?|kit|cajas|frascos|rollos)\b|\+|\d\s*[x×]\s*\d+\s*(?:tabletas?|capsulas?|sobres?)", nombre):
        return "", "", None

    if re.search(r"\bvendas?\b", nombre):
        medida = re.search(numero+r"\s*cm\s*[x×]\s*"+numero+r"\s*(cm|mts?|m|metros?)\b", nombre+" ; "+descripcion)
        if not medida:
            return "", "", None
        tipo = next((t for t in ["autoadherible", "tubular", "enyesada", "elastomedic", "elastica"] if t in nombre), "")
        if tipo:
            largo = valor(medida[2]) / (100 if medida[3]=="cm" else 1)
            return f"Venda {tipo} · ancho {medida[1]} cm", "m", largo
        return "", "", None

    if re.search(r"\bgasas?\b", nombre):
        dimensiones = re.search(numero+r"\s*(?:cm)?\s*[x×]\s*"+numero+r"\s*cm\b", nombre)
        unidades = re.search(r"(\d+)\s*(?:piezas?|pzs?|pzas?|gasas?)\b", nombre)
        if dimensiones and unidades:
            tipos = [t for t in ["no esteril","antiseptica","antimicrobiana","diabetico","absorbente","premium","esteril"] if t in nombre]
            tipo = " ".join(tipos) or "sin especificación adicional"
            return f"Gasa {tipo} · {dimensiones[1]} × {dimensiones[2]} cm", "pieza", int(unidades[1])
        return "", "", None

    # Mismo producto o activo reconocido, misma potencia, forma y variante.
    piezas = list(re.finditer(r"(\d+)\s*(tabletas?|tabs?|capsulas?|caps?|comprimidos?|grageas?|sobres?|piezas?|pzs?|pzas?)\b", nombre))
    dosis = list(re.finditer(numero+r"\s*mg\b", nombre))
    if len(piezas)==1:
        p = piezas[0]
        forma = p[2]
        if forma.startswith(("tab","comprimido")):
            forma = "tableta"
        elif forma.startswith("cap"):
            forma = "cápsula"
        elif forma.startswith(("pz","pieza")):
            forma = "pieza"
        base_nombre = nombre[:p.start()] + " " + nombre[p.end():]
        if len(dosis)==1 and forma in {"tableta","cápsula","gragea","grageas"} and not re.search(r"/\s*ml|\d\s*/\s*\d|[a-z]\s*/\s*[a-z]|cafeina|paracetamol.*ibuprofeno", nombre):
            potencia = valor(dosis[0][1])
            identidad = re.sub(numero+r"\s*mg\b", " ", base_nombre)
            # Unifica únicamente las variantes sencillas identificadas en el título.
            if re.match(r"^(aspirina analgesico|acido acetilsalicilico\s*\d)", nombre):
                identidad = "ácido acetilsalicílico"
            elif re.match(r"^(ibuprofeno\s*\d|motrin\s*\d|advil(?: analgesico)?\s*\d)", nombre):
                identidad = "ibuprofeno"
            variantes = [t for t in ["efervescente","masticable","prolongada","retard","protect","junior","advanced","liquida","recubierta","sublingual"] if t in nombre]
            declaracion = re.fullmatch(r"(?:principio activo|contiene)\s*:?\s*([a-z]+)\.?", descripcion)
            if declaracion and not re.search(r"\b(?:xr|lp|sr|cr|ap|forte|plus|duo|retard|advanced)\b", nombre):
                identidad = declaracion[1]
            identidad = re.sub(r"\b(?:marca del ahorro|farmacias del ahorro|caja con|caja|con|oral|tabletas|capsulas)\b", " ", identidad)
            identidad = re.sub(r"[^a-záéíóúñ0-9]+", " ", identidad).strip()
            clave = " ".join(identidad.split()) + f" · {potencia:g} mg · {forma} " + " ".join(variantes)
            return clave.strip(), "mg", int(p[1])*potencia
        return " ".join(base_nombre.split())+" · "+forma, forma, int(p[1])

    # Frascos y tubos con un solo contenido neto explícito.
    medidas = list(re.finditer(numero+r"\s*(ml|g|gr)\b", nombre))
    if len(medidas)==1 and not re.search(r"/|\b(?:ampolletas?|jeringas?|inyectable|tabletas?|capsulas?)\b|\d\s*x\s*\d",nombre):
        m = medidas[0]
        identidad = " ".join((nombre[:m.start()]+" "+nombre[m.end():]).split())
        return identidad, "ml" if m[2]=="ml" else "g", valor(m[1])
    return "", "", None

detalle = agrupados.copy()
def limpiar_precio(valor):
    texto = re.sub(r"MXN|\$|\s", "", str(valor), flags=re.I)
    if re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", texto):
        texto = texto.replace(",", "")
    elif re.fullmatch(r"\d+,\d{1,2}", texto):
        texto = texto.replace(",", ".")
    numero = pd.to_numeric(texto, errors="coerce")
    if pd.notna(numero) and 0 < round(numero, 2) < float("inf"):
        return round(numero, 2)
    return float("nan")

detalle["Precio_MXN"] = detalle["Precio_actual"].map(limpiar_precio)
# Una fila puede participar en varios grupos. Revisa los SKU repetidos en la base de origen.
sku = base_limpia["SKU"].astype(str).str.strip()
repetidos = set(sku[sku.duplicated(keep=False) & sku.ne("")])
detalle["Revision"] = ""
for condicion, motivo in [(detalle["SKU"].astype(str).str.strip().isin(repetidos), "Revisar SKU repetido"),
                          (detalle["SKU"].astype(str).str.strip().eq(""), "Falta SKU"),
                          (detalle.nombre_busqueda.eq(""), "Falta nombre"),
                          (detalle.Precio_MXN.isna(), "Precio inválido")]:
    detalle.loc[condicion, "Revision"] += motivo + ". "

medidas = [presentacion(n, d) for n, d in zip(detalle.nombre_busqueda, detalle.descripcion_busqueda)]
detalle[["Subgrupo","Unidad","Contenido"]] = pd.DataFrame(medidas, index=detalle.index, columns=["Subgrupo","Unidad","Contenido"])
valido = detalle.Revision.eq("") & detalle.Grupo.ne("Sin clasificar")
unitario = valido & detalle.Contenido.gt(0) & detalle.Subgrupo.ne("")
detalle["MXN_por_unidad"] = detalle.Precio_MXN.div(detalle.Contenido.where(unitario))
detalle["Estado"] = "Revisar presentación"
detalle.loc[detalle.Grupo.eq("Sin clasificar"),"Estado"] = "Sin clasificar"
detalle.loc[detalle.Revision.ne(""),"Estado"] = detalle.loc[detalle.Revision.ne(""),"Revision"]
costos = detalle.loc[unitario].groupby(["Grupo","Subgrupo","Unidad"])["MXN_por_unidad"]
conteo = costos.transform("count")
minimo = costos.transform("min")
detalle.loc[unitario,"Estado"] = "Comparado"
detalle.loc[conteo.index[conteo.eq(1)],"Estado"] = "Única opción"
ganan = conteo.ge(2) & detalle.loc[unitario,"MXN_por_unidad"].round(12).eq(minimo.round(12))
detalle.loc[ganan.index[ganan],"Estado"] = "Menor costo unitario"
resumen = detalle.loc[detalle.Estado.eq("Menor costo unitario"),
                      ["Grupo","Subgrupo","SKU","Nombre_producto","Precio_MXN","Unidad","Contenido","MXN_por_unidad"]]
envases = detalle.loc[valido].copy()
envases["SKU_en_grupo"] = envases.groupby("Grupo")["SKU"].transform("nunique")
envases = envases[envases.SKU_en_grupo >= 2]
sku_envase = set(envases.SKU)
sku_unitario = set(detalle.loc[detalle.Estado.isin(["Comparado","Menor costo unitario"]), "SKU"])
# Un segundo nivel de referencia comercial exige además el mismo formato declarado
# en el título. Puede haber ingredientes, dosis y cantidades diferentes.
def formato_comercial(nombre):
    for formato, patron in patrones_formato:
        if patron.search(nombre):
            return formato
    return ""

patrones_formato = [
    (formato, re.compile(r"(?<![a-z])(?:"+"|".join(re.escape(normalizar(t)) for t in terminos)+r")(?![a-z])"))
    for formato, terminos in formatos.items()
]
mapa_formato = {nombre: formato_comercial(nombre) for nombre in envases.nombre_busqueda.unique()}
envases["Formato_envase"] = envases.nombre_busqueda.map(mapa_formato)
pares_formato = envases.loc[envases.Formato_envase.ne("") & ~envases.Grupo.str.startswith("Presentación · ")].copy()
pares_formato["Opciones"] = pares_formato.groupby(["Grupo","Formato_envase"]).SKU.transform("nunique")
pares_formato = pares_formato.loc[pares_formato.Opciones.ge(2)]
print(f"De los SKU con precio de envase comparable, {pares_formato.SKU.nunique():,} también comparten tema o término y formato con otro SKU.")
sku_fuente1 = set(envases.loc[envases.Fuente_asignacion.eq("Descripción corta"),"SKU"])
print(f"SKU únicos con precio de envase comparable: {len(sku_envase):,} (meta: 5,000).")
print(f"Con grupo comparable por descripción: {len(sku_fuente1):,}; adicionales por palabras clave: {len(sku_envase-sku_fuente1):,}.")
print(f"SKU únicos con comparación por unidad dentro de un subgrupo de presentación: {len(sku_unitario):,}.")
envases = envases[envases.Precio_MXN.eq(envases.groupby("Grupo").Precio_MXN.transform("min"))]
print("Menor precio del envase entre las coincidencias de cada grupo (pueden tener contenidos distintos):")
display(envases[["Grupo","Nombre_producto","Precio_MXN"]])
minimos_formato = pares_formato.groupby(["Grupo","Formato_envase"]).Precio_MXN.transform("min")
mejor_formato = pares_formato.loc[pares_formato.Precio_MXN.eq(minimos_formato)]
print("Menor desembolso por grupo y formato (se conservan empates):")
display(mejor_formato[["Grupo","Formato_envase","Nombre_producto","Precio_MXN","Opciones"]])
print("Menor costo por unidad en subgrupos con dos o más opciones; se conservan los empates:")
display(resumen)
# Activa esta opción si quieres una copia del resultado en Excel.
EXPORTAR = False
if EXPORTAR:
    salida = Path.cwd() / "resultado_por_grupos.xlsx"
    if salida.resolve() == ruta.resolve():
        raise ValueError("Elige un nombre de salida diferente al de la base.")
    with pd.ExcelWriter(salida, engine="openpyxl") as libro:
        pares_formato.to_excel(libro, sheet_name="Grupo y formato", index=False)
        resumen.to_excel(libro, sheet_name="Menor costo unitario", index=False)
        envases.to_excel(libro, sheet_name="Menor precio de envase", index=False)
        detalle.drop(columns=["nombre_busqueda","descripcion_busqueda"]).to_excel(libro, sheet_name="Detalle", index=False)
    print(f"Guardado en {salida}")

De los SKU con precio de envase comparable, 3,780 también comparten tema o término y formato con otro SKU.
SKU únicos con precio de envase comparable: 5,224 (meta: 5,000).
Con grupo comparable por descripción: 4,693; adicionales por palabras clave: 531.
SKU únicos con comparación por unidad dentro de un subgrupo de presentación: 1,154.
Menor precio del envase entre las coincidencias de cada grupo (pueden tener contenidos distintos):


,Grupo,Nombre_producto,Precio_MXN
0,Accesorios íntimos,Sico Softlube Lubricante Intimo Pleasure Plus ...,168.0
29,Acetilcisteína,Acetilcisteína 600 mg 20 Tabs Marca del Ahorro,153.0
37,Aciclovir,Aciclovir 5% 5 g Marca del Ahorro,45.0
57,Acidez,Supacid 20 mg 7 tabletas,25.5
150,Acné,Kitoscell Gel 10 g,294.0
...,...,...,...
17724,Vitamina C,Suerox Adulto 8 Iones Fresa-Kiwi 630 ml,22.0
17780,Vitamina D,Suerox Vitamins Energy Bebida Hidratante Manza...,22.0
17840,Vértigo y mareos,Dimenhidrinato 50 mg 24 Caps Marca del Ahorro,68.0
17856,Ácido fólico,Ácido Fólico 5 mg 20 tabletas Marca del Ahorro,23.0


Menor desembolso por grupo y formato (se conservan empates):


,Grupo,Formato_envase,Nombre_producto,Precio_MXN,Opciones
2,Accesorios íntimos,Geles,K-Y Gel Lubricante Íntimo Envase 50 gr,180.00,2
29,Acetilcisteína,Tabletas,Acetilcisteína 600 mg 20 Tabs Marca del Ahorro,153.00,6
38,Aciclovir,Cremas,Cicloferon 5% Crema 2 g,66.60,8
43,Aciclovir,Tabletas,Aciclovir 200 mg Oral 25 tabletas Marca del Ah...,280.00,4
45,Aciclovir,Aerosoles,Cicloferon 5% en Spray 50 ml,661.00,2
...,...,...,...,...,...
17865,Ácido fólico,Suspensiones,Ferricol Pediátrico Suspensión 45 ml,400.00,2
17878,Úlceras digestivas,Suspensiones,Estomaquil Exper 3 Antiácido 240 ml Suspensión,123.25,2
17880,Úlceras digestivas,Geles,Gelan Plus Gel 40 ml 4 Frascos,180.00,2
17881,Úlceras digestivas,Cápsulas,Amably 24 hrs 20 mg - Frasco con 7 cápsulas,227.00,6


Menor costo por unidad en subgrupos con dos o más opciones; se conservan los empates:


,Grupo,Subgrupo,SKU,Nombre_producto,Precio_MXN,Unidad,Contenido,MXN_por_unidad
7,Accesorios íntimos,k-y gel lubricante intimo envase,7891010005047,K-Y Gel Lubricante Íntimo Envase 100 gr,305.00,g,100.0,3.050000
29,Acetilcisteína,acetilcisteina · 600 mg · tableta,7506472849787,Acetilcisteína 600 mg 20 Tabs Marca del Ahorro,153.00,mg,12000.0,0.012750
38,Aciclovir,cicloferon 5% crema,7501299303047,Cicloferon 5% Crema 2 g,66.60,g,2.0,33.300000
43,Aciclovir,aciclovir · 200 mg · tableta,7502223703544,Aciclovir 200 mg Oral 25 tabletas Marca del Ah...,280.00,mg,5000.0,0.056000
55,Acidez,riopan gel oral · sobres,7501092793045,Riopan Gel Oral 20 sobres,286.00,sobres,20.0,14.300000
...,...,...,...,...,...,...,...,...
17769,Vitamina C,nikzon · tableta,650240001314,Nikzon 90 tabletas,525.60,tableta,90.0,5.840000
17815,Vitamina D,midenar 4000 u.i. · cápsula,7501300450708,MiDenar 4000 U.I. 60 Caps,239.00,cápsula,60.0,3.983333
17816,Vitamina D,"morebon 5,000 ui oral · tableta",7501249600943,"Morebon 5,000 UI Oral 60 Tabs",249.00,tableta,60.0,4.150000
17821,Vitamina D,"wise tree vitamina d3 5,000 ui suplemento alim...",7506531000029,"Wise Tree Vitamina D3 5,000 Ui Suplemento Alim...",272.85,cápsula,90.0,3.031667
